# TASK 2 Build a Multi-Turn Conversational Layer on Top of LLMOPT

In [1]:
!nvidia-smi

Tue Jun  2 13:50:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cloning the Repository

In [2]:
!git clone https://github.com/caigaojiang/LLMOPT

fatal: destination path 'LLMOPT' already exists and is not an empty directory.


## Installing Dependencies

In [30]:
!pip install transformers==4.42.0 \
             accelerate==0.27.2 \
             tiktoken==0.7.0 \
             einops==0.7.0 \
             transformers_stream_generator==0.0.4 \
             peft==0.11.1 \
             trl==0.9.6 \
             Jinja2==3.1.4 \
             bitsandbytes \
             "numpy<=2.0.0 "\
             "outlines<1.0.0 " \
             pydantic \
             pyomo
# Install the Linux system-level math solvers that Pyomo relies on
!apt-get install -y coinor-cbc glpk-utils

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.4/90.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.4/935.4 kB 29.9 MB/s eta 0:00:00
  Attempting uninstall: outlines_core
    Found existing installation: outlines_core 0.2.14
    Uninstalling outlines_core-0.2.14:
      Successfully uninstalled outlines_core-0.2.14
  Attempting uninstall: outlines
    Found existing installation: outlines 1.3.0
    Uninstalling outlines-1.3.0:
      Successfully uninstalled outlines-1.3.0


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
coinor-cbc is already the newest version (2.10.7+ds1-1).
glpk-utils is already the newest version (5.0-1).
0 upgraded, 0 newly installed, 0 to remove and 141 not upgraded.


## Load Necessary modules

In [31]:
import os
import json
import sys


import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    pipeline
)

import peft
import trl
import pydantic
import outlines
from typing import List, Dict, Union, Literal
from pyomo.environ import *
import pyomo.environ as aml

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Current GPU: Tesla T4


In [5]:
# Load the necessary prompt modules from the official LLOMPT repo to build prompts
module_path = os.path.abspath('LLMOPT/prompts')
print(f"module_path: {module_path}")
# Add to sys.path if not already present
if module_path not in sys.path:
    sys.path.append(module_path)

import generate_prompt
import self_correction_prompt

module_path: /kaggle/working/LLMOPT/prompts


## Load the Model
Use quantization to compress the model

In [7]:
# Configure the 4-bit quantization layout to protect cloud VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "ant-opt/LLMOPT-Qwen2.5-14B"

# Download and cache the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Download, quantize, and load the 14B model safely
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True        # Forces shard-by-shard loading directly into RAM/VRAM
)

print("done")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

...

model-00011-of-00012.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00012-of-00012.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

done


## Load the industryor Dataset

In [13]:
dataset_path = "./LLMOPT/data/testset/industryor.jsonl"

# open and laod the dataset (json file)
with open(dataset_path, "r") as f:
    test_problems = [json.loads(line) for line in f]

print(f"Successfully loaded {len(test_problems)} test items from industryor.")
print(f"Sample test problem:\n{json.dumps(test_problems[0], indent=1)}")

Successfully loaded 100 test items from industryor.
Sample test problem:
{
 "question": "The Zhang family has 6 children, Harry, Hermione, Ron, Fred, George, and Ginny. The cost of taking Harry is $1200, Hermione is $1650, Ron is $750, Fred is $800, George is $800, and Ginny is $1500. Which children should the couple take to minimize the total cost of taking the children?\n\nThey can take a maximum of 4 children on the upcoming trip.\n\nGinny is the youngest, so the Zhang family will definitely take her.\n\nIf the couple takes Harry, they will not take Fred because Harry doesn't get along with him.\n\nIf the couple takes Harry, they will not take George because Harry doesn't get along with him.\n\nIf they take George, they must also take Fred.\n\nIf they take George, they must also take Hermione.\n\nAlthough this will cost them a lot of money, the Zhang family has decided to take at least three children.",
 "answer": 3050,
 "ori": "6_industryor",
 "index": 1
}


## Testing and Evaluating the Model

### Copy the Necessary Functions from the official LLOMPT REPO And Create New functions to do the Self Correction Loop

In [14]:
import subprocess
# inference to get five elements
def infer_five_elem(question):
    messages = [
        {"role": "user", "content": generate_prompt.Q2F(question)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    response = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")

    if "```text" in response:
        return response.split("```text")[1].split("```")[0]
    elif "```plaintext" in response:
        return response.split("```plaintext")[1].split("```")[0]
    elif "```" in response:
        return response.split("```")[1].split("```")[0]
    else:
        return None


# inference to get pyomo python code
def infer_code(five_elem):
    messages = [
        {"role": "user", "content": generate_prompt.F2C(five_elem)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans.split("```python")[1].split("```")[0].replace('print("\\\\\n', 'print("').replace('print(f"\\\\\n', 'print(f"')


# execute the code
def test_code(code_str):
    code_path = f"./test.py"
    with open(code_path, "w") as f1:
        f1.write(code_str)

    ans = subprocess.run(f"python {code_path}", shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    # return answer logs, error code
    return str(ans.stdout.decode('gbk', errors='ignore')), str(ans.stderr.decode('gbk', errors='ignore'))

In [15]:
# this one is not from the official repo
def self_correct(ques, five, code, output, error):
    messages = [
        {"role": "user", "content": self_correction_prompt.self_correction(ques, five, code, output, error)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans


def self_correct_analysis_five_elem_prompt(analysis, ques, five):
    return f"""For the following optimization problem, modeling is performed, and pyomo code is generated and executed based on the modeling. Please fix the modeling and code based on the analysis.

The analysis is as follows. 

{analysis}

The problem is as follows.

{ques}

The five-element formulation is as follows.

{five}

Based on the analysis. Please write the corresponding five-element model. Please use LaTeX and ``` plain text environment to complete the following template to model the above optimization problem into five elements: 

```
## Sets: 
[You need to fill in]

## Parameters: 
[You need to fill in]

## Variables: 
[You need to fill in]

## Objective: 
[You need to fill in]

## Constraints: 
[You need to fill in]
```
"""

    
def self_correct_analysis_code_prompt(analysis, five, code):
    return f"""For the following optimization problem, modeling is performed, and pyomo code is generated and executed based on the modeling. Please fix the modeling and code based on the analysis.

The analysis is as follows. 

{analysis}

The five-element formulation is as follows.

{five}

The code is as follows.

{code}

Based on the analysis. Please write the corresponding Pyomo code. Please add `from pyomo.environ import *` at the beginning of your code (You can add other `import` as well). Please print the optimal solution and the value of the objective function. Please do not output the running log. You need to write it in the form of a class and add a main function: 

```python
[write your code here]
```
"""

def self_correct_analysis_five_elem(analysis, ques, five):
    messages = [
        {"role": "user", "content": self_correct_analysis_five_elem_prompt(analysis, ques, five)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    response = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")

    if "```text" in response:
        return response.split("```text")[1].split("```")[0]
    elif "```plaintext" in response:
        return response.split("```plaintext")[1].split("```")[0]
    elif "```" in response:
        return response.split("```")[1].split("```")[0]
    else:
        return None

def self_correct_analysis_code(analysis, five, code):
    messages = [
        {"role": "user", "content": self_correct_analysis_code_prompt(analysis, five, code)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans.split("```python")[1].split("```")[0].replace('print("\\\\\n', 'print("').replace('print(f"\\\\\n', 'print(f"')

### Running the model

In [16]:
TEST_LIMIT = 20  # Evaluate 10 test problems
MAX_CORRECTION_ATTEMPTS = 3 
START_INDEX = 20
END_INDEX = TEST_LIMIT + START_INDEX

print(f"Launching real-time evaluation over {TEST_LIMIT} problems...")

real_time_data = test_problems[START_INDEX:END_INDEX]

successful_oneshot_count = 0
successful_corrected_count = 0
for i, item in enumerate(real_time_data):
    print(f"\nProcessing Problem {i+1}/{TEST_LIMIT}...")
    problem_description = item['question']
    problem_answer = str(item['answer'])
    print('question = ' + problem_description)
    print("answer = " + problem_answer)
    # --- PHASE 1: FIVE-ELEMENT FORMULATION ---  
    five_element_text = infer_five_elem(problem_description)
    
    # --- PHASE 2: INITIAL CODE GENERATION ---
    code = infer_code(five_element_text)
   
    # --- PHASE 3: EXECUTION & AUTO-TESTING SELF CORRECTION LOOP ---
    attempt = 0
    solved_successfully = False
    oneshot_win = False
    wrong_code = []

    while attempt < MAX_CORRECTION_ATTEMPTS:
        # Clean up any leftover markdown block syntax before execution
        run_res = test_code(code)

        print(f"attempt {attempt} output: {run_res[0]}")
        check_answer = problem_answer in run_res[0]

        if not run_res[1] and check_answer:
            solved_successfully = True
            if attempt == 0:
                oneshot_win = True
            break
        else:
            attempt += 1
            wrong_code.append(code)
            # Run Self-Correction Generation Pass
            correction_prompt_input = {
                'ques': problem_description, 
                'five': five_element_text, 
                'code': code, 
                'output': run_res[0], 
                'error': run_res[1]
            }
            
            self_correct_output = self_correct(**correction_prompt_input)
            five_element_text = self_correct_analysis_five_elem(self_correct_output, problem_description, five_element_text)
            code = self_correct_analysis_code(self_correct_output, five_element_text, code)

    if oneshot_win:
        successful_oneshot_count += 1
    if solved_successfully:
        successful_corrected_count += 1
    
    print(f"↳ Result - OneShot Win: {oneshot_win} | Final Success: {solved_successfully} (Attempts: {attempt})")
    if wrong_code:
        for i, wc in enumerate(wrong_code):
            print("================================================================")
            print(f"Attempt {i}:\n\n{wc}")
            print("================================================================")

# --- RESULTS SUMMARY (Indentation Fixed) ---
print("\n==================== ACTUAL TASK 1 METRICS ====================")
print(f"Total Problems Processed: {TEST_LIMIT}")
print(f"Accuracy WITHOUT Self-Correction: {(successful_oneshot_count/TEST_LIMIT)*100:.2f}%")
print(f"Accuracy WITH Self-Correction: {(successful_corrected_count/TEST_LIMIT)*100:.2f}%")
print("================================================================")

Launching real-time evaluation over 20 problems...

Processing Problem 1/20...
question = Red Star Plastic Factory produces 6 types of plastic containers, with their capacities, demands, and variable costs shown in Table 5-11.
Table 5-11
\begin{tabular}{c|c|c|c|c|c|c}
\hline Container Code & 1 & 2 & 3 & 4 & 5 & 6 \
\hline Capacity $(\mathrm{cm}^3)$ & 1500 & 2500 & 4000 & 6000 & 9000 & 12000 \
Demand/units & 500 & 550 & 700 & 900 & 400 & 300 \
Variable Cost/(¥/unit) & 5 & 8 & 10 & 12 & 16 & 18 \
\hline
\end{tabular}

Each type of container is produced using different dedicated equipment, with a fixed cost of ¥1200. When the quantity of a certain container cannot meet the demand, containers with larger capacities can be used as substitutes. How should production be organized to minimize the total cost while meeting the demand?
answer = 43700


The attention mask is not set and cannot be inferred from input because pad token is same as eos token.As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


attempt 0 output: 
attempt 1 output: Optimal Production Numbers:
Container 1: 500 units
Container 2: 550 units
Container 3: 700 units
Container 4: 900 units
Container 5: 400 units
Container 6: 300 units
Total Cost: 37700.0

attempt 2 output: Optimal Production Numbers:
Container 1: 500 units
Container 2: 550 units
Container 3: 700 units
Container 4: 900 units
Container 5: 400 units
Container 6: 300 units
Total Cost: 37700.0

↳ Result - OneShot Win: False | Final Success: False (Attempts: 3)
Attempt 0:



from pyomo.environ import *

class ContainerProductionModel:
    def __init__(self):
        self.model = ConcreteModel()
        
        # Sets
        self.C = [1, 2, 3, 4, 5, 6]

        # Parameters
        self.Capacity = {1: 1500, 2: 2500, 3: 4000, 4: 6000, 5: 9000, 6: 12000}
        self.Demand = {1: 500, 2: 550, 3: 700, 4: 900, 5: 400, 6: 300}
        self.VariableCost = {1: 5, 2: 8, 3: 10, 4: 12, 5: 16, 6: 18}
        self.FixedCost = 1200

        # Variables
        self.mo

KeyboardInterrupt: 

In [ ]:
class FiveElementSchema(BaseModel):
    sets: Dict[str, List[str]] = Field(description="Dictionary mapping sets to arrays.")
    parameters: Dict[str, float] = Field(description="Flat numerical key-value constants.")
    variables: List[str] = Field(description="List of decision variables.")
    objective: str = Field(description="Optimization objective expression.")
    constraints: List[str] = Field(description="Array of clear mathematical boeundaries.")
    current_status: Literal["COMPLETE", "INCOMPLETE"]
    next_question: str

In [37]:
# 1. Monkeypatch the missing method onto your tokenizer instance
if not hasattr(tokenizer, 'get_chat_template'):
    # Outlines expects this method; we point it directly to the chat_template property
    tokenizer.get_chat_template = lambda *args, **kwargs: getattr(tokenizer, "chat_template", None)

# 2. Safety check: Ensure Qwen's ChatML template string is explicitly set if empty
if not tokenizer.chat_template:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{'<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>\\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "{{'<|im_start|>assistant\\n'}}"
        "{% endif %}"
    )

# 3. Now initialize the structural mask safely
outlines_model = outlines.from_transformers(model, tokenizer)
print("done")

done


In [42]:
def modified_Q2F(ques):
    ques = generate_prompt.bound_symbol + ques + generate_prompt.bound_symbol

    dialogue_instruction = """You are acting as an interactive Operations Research Consultant. The user has provided an initial problem statement, followed by a running conversation history. 
Your job is to progress towards building a mathematically sound five-element optimization model by eliciting missing details or checking for logical contradictions.

CRITICAL LOGICAL RULES:
1. GROUNDING RULE: If information, sets, parameters, or numeric values are not explicitly stated by the user, you MUST mark them as [MISSING]. Do NOT invent, guess, or hallucinate arbitrary variables, costs, constraints, or numbers.
2. QUESTION ISOLATION RULE: When asking a clarifying question, base it strictly on what is missing. Do not invent anything if the user has not explicitly mentioned them. And Do not ask anything the user has explicitly mentioned.
"""
    
    modified_five_suffix = """
STRICT STATUS DEFINITIONS:
- If any elements or numerical parameters required to build the model are unknown or marked MISSING, you MUST set Status under Next Action to INCOMPLETE.
- If and only if ALL five elements are fully collected, clear, and have zero MISSING tags, you MUST set Status under Next Action to COMPLETE. 
- If you detect that the user's requirements or stated constraints are mathematically impossible or directly contradict each other, set Status to CONFLICT.

STRICT ANSWER FORMAT:
- Please analyze the problem description and write the corresponding five-element model. 
- Please use json and {} in string format to complete the following template to model the above optimization problem into five elements: 

"""

    return generate_prompt.five_description_complex + generate_prompt.ques_description_five + ques + modified_five_suffix

In [45]:
def modified_infer_five_elem(question):
    """
    Wraps user text in Qwen's ChatML template and forces a structured 
    dictionary output matching the LLMOPT five-element layout.
    """
    generate_prompt_question = modified_Q2F(question)
    # Build a crisp prompt matching Qwen's training format
    prompt = f"""<|im_start|>system
You are an assistant acting as an interactive Operations Research Consultant. The user has provided an initial problem statement, followed by a running conversation history. 
Your job is to progress towards building a mathematically sound five-element optimization model by eliciting missing details or checking for logical contradictions.
<|im_end|>
<|im_start|>user
Latest Message: {question}
<|im_end|>
<|im_start|>assistant
"""

    # Pass prompt and schema directly to the Outlines model instance
    # This intercepts the token loop and forces 100% syntactically valid JSON
    structured_obj = outlines_model(prompt, FiveElementSchema)
    
    # Convert the Pydantic object back into a standard Python dictionary 
    # using your exact key aliases for your downstream evaluation pipeline
    return structured_obj.model_dump(by_alias=True)

In [46]:
y = modified_infer_five_elem('help me schedule my delivery drivers this week"')

ValueError: Input length of input_ids is 110, but `max_length` is set to 20. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.

In [ ]:
print(y)